In [1]:
# ============================================================
# Silver: limpieza y conformación de generación EERSA
# - Lee de bronze_generacion_diaria (Lakehouse Bronze)
# - Filtra ruido, deduplica, tipifica
# - Clasifica fuentes (generacion_propia / interconexion / agregado / flujo_mem)
# - Escribe silver_generacion_diaria + silver_generacion_quarantine
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, DateType, StringType
from pyspark.sql.window import Window

# Leer Bronze (cross-lakehouse)
df_bronze = spark.sql("SELECT * FROM lh_bronze_eersa.dbo.bronze_generacion_diaria")
print(f"Bronze input: {df_bronze.count()} filas")

# 1. Filtrar métricas válidas (eliminar Unnamed, KWh.1, KWh.2, etc.)
metricas_validas = ["KW", "E.Bruta kWh", "C.Int kWh", "E.Neta kWh", "E.Neta Kwh", "KWh", "KW-H"]
df = df_bronze.filter(F.col("metrica").isin(metricas_validas))

# 2. Normalizar métricas (E.Neta Kwh con K mayúscula → E.Neta kWh)
df = df.withColumn(
    "metrica",
    F.when(F.lower(F.col("metrica")) == "e.neta kwh", F.lit("E.Neta kWh"))
     .otherwise(F.col("metrica"))
)

# 3. Casteo de valor a Double, capturando los que fallan
df = df.withColumn("valor_num", F.col("valor").cast(DoubleType()))

# 4. Separar válidos y cuarentena
df_quarantine = df.filter(F.col("valor_num").isNull() & F.col("valor").isNotNull())
df_clean = df.filter(F.col("valor_num").isNotNull())

print(f"Cuarentena (no numéricos): {df_quarantine.count()} filas")
print(f"Limpios: {df_clean.count()} filas")

# 5. Clasificar tipo de fuente
df_clean = df_clean.withColumn(
    "tipo_fuente",
    F.when(F.col("planta").isin("ALAO", "RIO BLANCO", "C.NIZAG"), "generacion_propia")
     .when(F.col("planta") == "S.N.I.", "interconexion")
     .when(F.col("planta") == "TOTAL EERSA", "agregado")
     .when(F.col("planta").contains("MEM"), "flujo_mem")
     .otherwise("desconocido")
)

# 6. Redondear valores a 3 decimales
df_clean = df_clean.withColumn("valor_num", F.round("valor_num", 3))

# 7. Deduplicar por (fecha, planta, metrica)
ventana = Window.partitionBy("fecha", "planta", "metrica").orderBy(F.col("_ingested_at").desc())
df_clean = df_clean.withColumn("rn", F.row_number().over(ventana)) \
                   .filter(F.col("rn") == 1) \
                   .drop("rn")

# 8. Selección final con tipos correctos
df_silver = df_clean.select(
    F.col("fecha").cast(DateType()).alias("fecha"),
    F.col("planta").cast(StringType()).alias("planta"),
    F.col("tipo_fuente").cast(StringType()).alias("tipo_fuente"),
    F.col("metrica").cast(StringType()).alias("metrica"),
    F.col("valor_num").alias("valor"),
    F.col("anio").cast(IntegerType()).alias("anio"),
    F.col("mes").cast(IntegerType()).alias("mes"),
    F.col("_source_file").alias("_source_file"),
    F.col("_ingested_at").alias("_bronze_ingested_at"),
    F.current_timestamp().alias("_silver_processed_at")
)

# Cuarentena
df_quarantine_out = df_quarantine.select(
    "fecha", "planta", "metrica", "valor",
    F.lit("valor_no_numerico").alias("razon_cuarentena"),
    "_source_file", "_ingested_at"
)

# Escribir tablas en Silver (default lakehouse)
df_silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .format("delta").saveAsTable("silver_generacion_diaria")

df_quarantine_out.write.mode("overwrite").option("overwriteSchema", "true") \
    .format("delta").saveAsTable("silver_generacion_quarantine")

print(f"\n✓ silver_generacion_diaria: {df_silver.count()} filas")
print(f"✓ silver_generacion_quarantine: {df_quarantine_out.count()} filas")

display(df_silver.limit(20))


StatementMeta(, f44f2db9-0c59-466a-afbf-da6ee65ff584, 3, Finished, Available, Finished, False)

Bronze input: 7695 filas
Cuarentena (no numéricos): 0 filas
Limpios: 5840 filas

✓ silver_generacion_diaria: 5840 filas
✓ silver_generacion_quarantine: 0 filas


SynapseWidget(Synapse.DataFrame, fbbe30af-e67f-4fa1-beb5-091907403ad6)